<img src=https://aggis.org/images/aggis.ico> 

# World Bank API Tool
## `worldbankapi.ipynb`
### aggis.org/files/notebooks/.
Adapted from assignment submission for UCLA GEOG 412: Programming for Geospatial Data Science, Unit 3.
<hr/>

#### 📦 Packages

In [ ]:
!pip install geopandas mapclassify

import requests
import pandas as pd
import geopandas as gpd
import numpy as np
import mapclassify
import matplotlib.pyplot as plt
import matplotlib.colors as cl
import ipywidgets as widgets
from IPython.display import display, clear_output

#### 🔨 Tool: World Bank API Economic & Demographic Data Mapping Tool

In [ ]:
# variable selections, global variables

year_options = [2020,2019,2018,2016,2009]
variable_options = [('Urban Population, %/total','SP.URB.TOTL.IN.ZS'),
  ('Rural Population, %/total','SP.RUR.TOTL.ZS'),
  ('Male labor force participation, %/male pop','SL.TLF.CACT.MA.ZS'),
  ('Female labor force participation, %/female pop','SL.TLF.CACT.FE.ZS'),
  ('Poverty headcount ratio at national poverty lines, %/population','SI.POV.NAHC'),
  ('crude birth rate, /1k people','SP.DYN.CBRT.IN'),
  ('crude death rate, /1k people','SP.DYN.CDRT.IN'),
  ('government expenditure on education, %/GDP','SE.XPD.TOTL.GD.ZS'),
  ('military expenditure, %/GDP','MS.MIL.XPND.GD.ZS'),
  ('research and development expenditure, %/GDP','GB.XPD.RSDV.GD.ZSN')]
color_options = ['Greys','Reds','Greens','Blues','Purples']

# dropdown selectors

year_dropdown = widgets.Dropdown(options=year_options,description='Year:',disabled=False)
variable_dropdown = widgets.Dropdown(options=variable_options,description='Variable:',disabled=False)
color_dropdown = widgets.Dropdown(options=color_options,description='Color:',disabled=False,)

# grid

grid = widgets.GridspecLayout(1,3,height='60px')
grid[0,0] = year_dropdown
grid[0,1] = variable_dropdown
grid[0,2] = color_dropdown

# on change event / function

def on_change(event=None):
  clear_output()
  countries = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
  clear_output()
  print("Selected year: ",year_dropdown.value)
  print("Selected variable: ",variable_dropdown.value)
  print("Selected color scheme: ",color_dropdown.value)
  display(grid)
  r_url = 'https://api.worldbank.org/v2/country/all/indicator/{}?format=json&per_page=500&date={}'.format(variable_dropdown.value,year_dropdown.value)
  r = requests.get(r_url)
  r_json = r.json()
  r_json = r_json[1]

  df = pd.DataFrame(r_json)
  df_subset = df[['countryiso3code','value']]
  countries_joined = countries.merge(df_subset,left_on='iso_a3',right_on='countryiso3code')

  fig, ax = plt.subplots(figsize=(16,6))
  ax.set_title(str(variable_dropdown.value) + ', ' + str(year_dropdown.value),fontsize=24) # not sure how to get title to read as description instead of value, i.e. 'Total Population' vs 'SP.POP.TOTL'
  ax.set_axis_off()
  countries_joined.plot(
      ax=ax,
      column='value',
      cmap=color_dropdown.value,
      legend=True,
      figsize=(16,6))

  countries_joined.hist(
      column='value',
      figsize=(14,6))

# observe

year_dropdown.observe(on_change,names='value')
variable_dropdown.observe(on_change,names='value')
color_dropdown.observe(on_change,names='value')

# start

display(grid)